# **Tính điểm xem video của người dùng**

In [1]:
import pandas as pd
import gc

# ============================================
# BƯỚC 1: ĐỌC DỮ LIỆU USER-VIDEO
# ============================================
print("Đang đọc user-video segments...")
df1 = pd.read_parquet('/kaggle/input/d-liu-gn-nhn/user-video-seg1.parquet')
df2 = pd.read_parquet('/kaggle/input/d-liu-gn-nhn/user-video-seg2.parquet')

user_video_df = pd.concat([df1, df2], ignore_index=True)
del df1, df2
gc.collect()

print(f"User-video shape: {user_video_df.shape}")

# Explode seq và lấy video_id ngay lập tức
print("Đang explode và extract video_id...")
user_video_df = user_video_df.explode('seq', ignore_index=True)

# Extract video_id trực tiếp, không giữ cột segments
user_video_df['video_id'] = user_video_df['seq'].apply(
    lambda x: x.get('video_id') if isinstance(x, dict) else None
)

# Chỉ giữ user_id và video_id, loại bỏ seq/segments
user_video_df = user_video_df[['user_id', 'video_id']].dropna().drop_duplicates()

# Chuyển sang category ngay
user_video_df['user_id'] = user_video_df['user_id'].astype('category')
user_video_df['video_id'] = user_video_df['video_id'].astype('category')

print(f"Sau khi xử lý: {user_video_df.shape}")
print(f"Memory usage:\n{user_video_df.memory_usage(deep=True)}")
gc.collect()

# ============================================
# BƯỚC 2: ĐỌC DỮ LIỆU COURSE
# ============================================
print("\nĐang đọc course data...")
users_course_labeling_df = pd.read_parquet('/kaggle/input/d-liu-gn-nhn/user_course_use_labeling.parquet')
users_course_labeling_df['user_id'] = users_course_labeling_df['user_id'].astype('category')
users_course_labeling_df['course_id'] = users_course_labeling_df['course_id'].astype('category')

resource_video_df = pd.read_parquet('/kaggle/input/d-liu-gn-nhn/resource_video_course.parquet')

# Lọc course có video point
course_proportion_df = users_course_labeling_df[['course_id', 'Video', 'Assignment', 'Exam']].drop_duplicates()
resource_video_df = resource_video_df.merge(
    course_proportion_df, 
    left_on='id', 
    right_on='course_id', 
    how='inner'
)

# Chỉ giữ các cột cần thiết
resource_video_df = resource_video_df[['id', 'video_id', 'video_counts']].rename(columns={'id': 'course_id'})
print(f"Resource video shape: {resource_video_df.shape}")

del course_proportion_df
gc.collect()

# ============================================
# BƯỚC 3: TẠO VIDEO-TO-COURSE MAPPING (TỐI ƯU)
# ============================================
print("\nĐang tạo video-to-course mapping...")
video_to_course = resource_video_df[['course_id', 'video_id']].explode('video_id')
video_to_course = video_to_course.dropna().drop_duplicates()

video_to_course['video_id'] = video_to_course['video_id'].astype('category')
video_to_course['course_id'] = video_to_course['course_id'].astype('category')

print(f"Video-to-course shape: {video_to_course.shape}")
print(f"Memory: {video_to_course.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================
# BƯỚC 4: MERGE VÀ TÍNH TOÁN (THEO CHUNKS)
# ============================================
print("\nĐang merge và tính watched counts...")

# Merge để tìm (user, course, video) watched
watched = user_video_df.merge(video_to_course, on='video_id', how='inner')
print(f"Watched shape: {watched.shape}")

del user_video_df, video_to_course
gc.collect()

# Drop duplicates để tránh đếm trùng
watched = watched[['user_id', 'course_id', 'video_id']].drop_duplicates()

# Đếm số video watched per (user, course)
print("Đang đếm watched videos...")
watched_counts = (
    watched.groupby(['user_id', 'course_id'], observed=True)
    .size()
    .rename('watched_videos')
    .reset_index()
)

print(f"Watched counts shape: {watched_counts.shape}")

del watched
gc.collect()

# ============================================
# BƯỚC 5: TÍNH PHẦN TRĂM
# ============================================
print("\nĐang tính watch percentage...")

# Lấy total video counts per course
course_video_counts = resource_video_df[['course_id', 'video_counts']].drop_duplicates()

# Merge để lấy tổng số video
result = watched_counts.merge(
    course_video_counts,
    on='course_id',
    how='inner'
)

del watched_counts, resource_video_df, course_video_counts, users_course_labeling_df
gc.collect()

# Tính phần trăm
result['watch_percent'] = result['watched_videos'] / result['video_counts']

# Giới hạn watch_percent <= 1 (nếu có lỗi dữ liệu)
result['watch_percent'] = result['watch_percent'].clip(upper=1.0)

print(f"\nKết quả cuối cùng: {result.shape}")
print(f"Memory usage: {result.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nSample results:")
print(result.head(10))

# Kiểm tra thống kê
print("\nThống kê watch_percent:")
print(result['watch_percent'].describe())

# Lưu kết quả
print("\nĐang lưu kết quả...")
result.to_parquet('/kaggle/working/user_video_scores.parquet', index=False)
print("Hoàn thành!")

Đang đọc user-video segments...
User-video shape: (310360, 2)
Đang explode và extract video_id...
Sau khi xử lý: (1981001, 2)
Memory usage:
Index       15848008
user_id     37141947
video_id    24925215
dtype: int64

Đang đọc course data...
Resource video shape: (2232, 3)

Đang tạo video-to-course mapping...
Video-to-course shape: (126778, 2)
Memory: 13.83 MB

Đang merge và tính watched counts...
Watched shape: (115724, 3)
Đang đếm watched videos...
Watched counts shape: (28643, 3)

Đang tính watch percentage...

Kết quả cuối cùng: (28643, 5)
Memory usage: 30.42 MB

Sample results:
      user_id  course_id  watched_videos  video_counts  watch_percent
0   U_1000454  C_2199449               1            41       0.024390
1   U_1002784  C_1756056               3            76       0.039474
2    U_100281   C_747033               3            49       0.061224
3  U_10033817   C_676642               3            50       0.060000
4   U_1004646   C_735164               1            61       

In [2]:
result

,user_id,course_id,watched_videos,video_counts,watch_percent
0,U_1000454,C_2199449,1,41,0.024390
1,U_1002784,C_1756056,3,76,0.039474
2,U_100281,C_747033,3,49,0.061224
3,U_10033817,C_676642,3,50,0.060000
4,U_1004646,C_735164,1,61,0.016393
...,...,...,...,...,...
28638,U_996997,C_735164,3,61,0.049180
28639,U_997041,C_735164,4,61,0.065574
28640,U_99753,C_1428968,7,49,0.142857
28641,U_997542,C_2066096,4,48,0.083333
